# 01 · EDA

**Part A (T1): raw inspection.** Look at the raw file *before* writing any cleaning rule. Every odd
thing you find here should become one of three things: a parsing fix, a cleaning rule, or a known
bias in `DATA_CARD.md`.

**Part B (T2): exploration of the cleaned data.** Comes later and loads `data/processed/listings.parquet`.

> Habit: when something looks odd, write the hypothesis down *before* you check it.

## Setup

In [ ]:
# setup and loading scaffold
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from fairprice.data import parse
from fairprice.paths import RAW_DIR

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

RAW_CSV = RAW_DIR / "kaggle" / "portugal_listinigs.csv"  # misspelled upstream

Load the file **as is**: no `dtype=`, no `parse_dates=`. You want to see what the raw file looks like,
not what you hope it looks like. `low_memory=False` makes pandas infer each column's type from the
whole column instead of from chunks. The default chunked mode can give one column different types
in different chunks.

In [ ]:
raw = pd.read_csv(RAW_CSV, low_memory=False)
# Parsed copy: snake_case names, proper dtypes, same rows (see src/fairprice/data.py).
# City -> municipality, Town -> parish, Type -> property_type.
df = parse(raw)
raw.shape, df.shape

## A0 · First look

In [ ]:
# one row per column: dtype, % missing, number of distinct values, a few example values
overview = pd.DataFrame(
    {
        "dtype": raw.dtypes.astype(str),
        "missing_%": raw.isna().mean().mul(100).round(1),
        "n_unique": raw.nunique(),
        "examples": [raw[c].dropna().unique()[:5].tolist() for c in raw.columns],
    }
).sort_values("missing_%", ascending=False)
overview

In [ ]:
raw.sample(10, random_state=0)

## A1 · Scope: apartments and houses for sale

- What are the distinct values of `Type`, and how many rows does each have?
- Which types belong in scope, and which are borderline? (Is a *Duplex* an apartment? A *Mansion* a house?)
- There is no sale/rent column. Could rentals still be in the file? How would you tell from the data alone?

In [ ]:
AREA_COLS = ["total_area", "living_area", "built_area", "lot_size"]


def type_profile(df: pd.DataFrame, area_col: str) -> pd.DataFrame:
    """Typical size and price per Type, plus signals that a type behaves like an apartment."""
    # Area 0 becomes NaN here so it cannot produce infinite EUR/m2. This is for display only,
    # so no rows are dropped.
    eur_m2 = df["price"] / df[area_col].where(df[area_col] > 0)
    g = df.assign(
        eur_m2=eur_m2,
        has_floor=df["floor"].notna(),
        zero_bath=df["bathrooms"].eq(0),
    ).groupby("property_type", dropna=False)
    return (
        pd.DataFrame(
            {
                "n": g.size(),
                "median_price": g["price"].median(),
                **{f"median_{c}": g[c].median() for c in AREA_COLS},
                f"median_eur_m2 [{area_col}]": g["eur_m2"].median(),
                "has_floor_%": g["has_floor"].mean() * 100,
                "zero_bath_%": g["zero_bath"].mean() * 100,
            }
        )
        .sort_values("n", ascending=False)
        .round(1)
    )


# Rerun with each column in AREA_COLS. Which one gives EUR/m2 values you believe?
type_profile(df, area_col="living_area")

In [ ]:
# Overlaid log-scale histograms. A second bump or a gap suggests two populations
# mixed together.
from matplotlib.ticker import FuncFormatter


def log_hist_by_type(
    df: pd.DataFrame, values: pd.Series, types: list[str], label: str, bins: int = 80
) -> plt.Axes:
    """density=True puts a 60-row type and a 70k-row type on the same vertical scale."""
    _fig, ax = plt.subplots(figsize=(10, 4))
    for t in types:
        v = values[
            df["property_type"].eq(t) & values.gt(0)
        ]  # NaN > 0 is False, so NaN drops out too
        ax.hist(np.log10(v), bins=bins, histtype="step", density=True, label=f"{t} (n={len(v):,})")
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{10**x:,.0f}"))
    ax.set_xlabel(f"{label} (log scale)")
    ax.set_ylabel("density")
    ax.legend()
    return ax

In [ ]:
# Pick the types to compare. Then try EUR/m2 with the area column you chose above.
log_hist_by_type(df, df["price"], ["Apartment", "House"], label="Price (EUR)");

In [ ]:
# Comparison set: the candidate scope plus the borderline types.
CANDIDATES = [
    "Apartment",
    "House",
    "Duplex",
    "Studio",
    "Estate",
    "Mansion",
    "Manor",
    "Other - Residential",
    "Farm",
]
cand = df[df["property_type"].isin(CANDIDATES)].copy()
SHOW = [
    "property_type",
    "price",
    "district",
    "municipality",
    "living_area",
    "total_area",
    "built_area",
    "lot_size",
    "bedrooms",
    "total_rooms",
    "bathrooms",
    "floor_label",
]

# Median EUR/m2 per type under each area column: with which area do the types look comparable?
eur_m2_by_area = pd.DataFrame(
    {
        a: (cand["price"] / cand[a].where(cand[a] > 0)).groupby(cand["property_type"]).median()
        for a in AREA_COLS
    }
).round(0)
eur_m2_by_area.join(cand["property_type"].value_counts().rename("n")).sort_values(
    "n", ascending=False
)

In [ ]:
# read the borderline types before deciding
for t in ["Other - Residential", "Estate", "Farm", "Studio", "Duplex"]:
    print(f"--- {t}")
    display(cand.loc[cand["property_type"].eq(t), SHOW].sample(8, random_state=0))

### Rentals?
A monthly rent is usually €300–5,000. If rentals are in the file, they show up as a cluster of
very low prices and a very low €/m².

In [ ]:
# rows per price band and type
bands = pd.cut(cand["price"], [0, 1, 1_000, 5_000, 10_000, 25_000, 50_000, np.inf], right=False)
pd.crosstab(cand["property_type"], bands)

In [ ]:
eur_m2_living = df["price"] / df["living_area"].where(df["living_area"] > 0)
log_hist_by_type(df, eur_m2_living, ["Apartment", "House"], label="EUR/m2 (living_area)");

In [ ]:
# the 20 cheapest candidate listings, in full
cand.nsmallest(20, "price")[SHOW]

## A2 · Types and parsing

- Which columns are `str`/`object` that should be numeric, boolean or dates?
- pandas warned about **mixed types** in `Floor`, `EnergyEfficiencyLevel`, `PublishDate`, `Garage`,
  `Elevator`, `ElectricCarsCharging` and `ConservationStatus`. What exactly is mixed in each?
- `Floor`: what values appear? How would you turn them into a number, and what should "Ground floor",
  "Basement" or "Top floor" become?

*(Parsing is implemented in `src/fairprice/data.py`.)*

Parsing lives in `src/fairprice/data.py` (types, renames, floor number, energy labels, text fixes).
Below: the result, and the most important structural finding.

In [ ]:
df.dtypes.to_frame("dtype").T

In [ ]:
# How floor labels were mapped to a number (NaN = layout/position, not a storey)
df[["floor_label", "floor"]].drop_duplicates().sort_values("floor", na_position="last")

### Two sources with different schemas
`garage`, `electric_cars_charging` and `energy_efficiency_level` are missing on exactly the same
rows. Those rows look like a different portal with a different set of fields. `source` below
is a **guess** inferred from that pattern. The dataset has no portal column.

In [ ]:
# Fill rate per column, per inferred source
source = df["garage"].isna().map({True: "A", False: "B"}).rename("source")
df.notna().groupby(source).mean().T.round(2)

## A3 · Impossible values: price, area, rooms

- `describe()` with extra percentiles (`[.001, .01, .5, .99, .999]`) on `Price`, the area columns,
  `NumberOfBedrooms`, `NumberOfBathrooms`, `TotalRooms`.
- Find the cheapest and the most expensive rows *per type* and read them in full. Typo, placeholder,
  wrong type, or real?
- There are **four** area columns: `TotalArea`, `LivingArea`, `BuiltArea`, `LotSize`. How do they relate
  row by row? Which one should the €/m² baseline use, and why? Look for rows where they disagree wildly.
- Compute €/m² with your chosen area. Which listings are extreme on €/m² but look normal on price
  and area on their own?

In [ ]:
# Distribution of the numeric columns for the candidate types
NUM = ["price", *AREA_COLS, "bedrooms", "total_rooms", "bathrooms", "wc"]
cand[NUM].describe(percentiles=[0.001, 0.01, 0.5, 0.99, 0.999]).T.round(1)

In [ ]:
# How the four area columns relate to living_area
living = cand["living_area"].where(cand["living_area"] > 0)
ratio = cand[AREA_COLS].div(living, axis=0)
pd.DataFrame(
    {
        "missing_%": cand[AREA_COLS].isna().mean() * 100,
        "zero_%": cand[AREA_COLS].eq(0).mean() * 100,
        "median_ratio_to_living": ratio.median(),
        "ratio_gt_3_%": ratio.gt(3).mean() * 100,
        "ratio_lt_0.5_%": ratio.lt(0.5).mean() * 100,
    }
).round(1)

In [ ]:
# Apartments whose total_area is 10x their living_area: what is total_area here?
odd = cand[cand["property_type"].eq("Apartment") & (cand["total_area"] > 10 * living)]
print(len(odd), "apartments")
odd[SHOW].sample(10, random_state=0)

In [ ]:
# EUR/m2 quantiles per area column, apartments and houses only
ah = cand[cand["property_type"].isin(["Apartment", "House"])]
q = [0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999]
pd.DataFrame({a: (ah["price"] / ah[a].where(ah[a] > 0)).quantile(q) for a in AREA_COLS}).round(0)

In [ ]:
# The most expensive listings and the most bedrooms
display(cand.nlargest(10, "price")[SHOW])
display(cand.nlargest(10, "bedrooms")[SHOW])
cand.nsmallest(10, "living_area")[SHOW]

In [ ]:
# Source A has total_rooms but no bedrooms. On source B rows that have both, how do they relate?
# (In Portugal "T2" means 2 bedrooms. Is total_rooms the T-number?)
b = cand[source.loc[cand.index].eq("B")].dropna(subset=["bedrooms", "total_rooms"])
print(
    f"{len(b):,} rows; total_rooms == bedrooms in {b['total_rooms'].eq(b['bedrooms']).mean():.0%}"
)
pd.crosstab(b["bedrooms"].clip(upper=8), b["total_rooms"].clip(upper=8))

## A4 · Duplicates

- How many rows are exact duplicates?
- Drop one column at a time from the duplicate check (for example `PublishDate`, or the energy fields)
  and watch how the count changes. Which columns make "the same home" look different?
- Pick one group of near-duplicates and read it. Is it the same home on two portals, a relisting, or
  different flats in the same building?

In [ ]:
# Exact duplicates, then the extra duplicates found when one column is ignored
n_exact = raw.duplicated().sum()
print(f"exact duplicate rows: {n_exact:,}")
leave_one_out = pd.Series(
    {c: raw.drop(columns=c).duplicated().sum() - n_exact for c in raw.columns}
).sort_values(ascending=False)
leave_one_out[leave_one_out > 0]

In [ ]:
# One candidate definition of "same home": the columns BOTH sources publish.
# Adding or removing key columns changes the count.
KEY = ["price", "property_type", "district", "municipality", "parish", "living_area"]
d = cand.drop_duplicates()  # exact duplicates first
dup = d.duplicated(subset=KEY, keep=False) & d["living_area"].notna()
groups = d[dup].groupby(KEY, dropna=False)
spans = groups["garage"].agg(lambda g: g.isna().nunique() > 1)  # mixes source A and B?
print(
    f"rows in near-duplicate groups: {dup.sum():,}; groups: {groups.ngroups:,}; "
    f"groups mixing both sources: {spans.sum():,}"
)

In [ ]:
# Read a few near-duplicate groups. Same home on two portals, a relisting, or different flats?
sample_keys = spans[spans].sample(3, random_state=0).index
for k in sample_keys:
    display(groups.get_group(k)[[*SHOW, "publish_date", "garage"]])

## A5 · Time

- Parse `PublishDate` into a scratch column (don't overwrite `raw`) and plot listings per year/quarter.
- Is missing `PublishDate` random, or does it go with certain districts, types or price levels?
- Which period would you keep, and which INE quarter would match it? (Decide later. Just collect the facts now.)

In [ ]:
# Listings per year (source B only; source A has no dates)
fig, ax = plt.subplots(figsize=(10, 3))
df["publish_date"].dt.year.value_counts().sort_index().plot.bar(ax=ax)
ax.set_ylabel("listings")
print(f"publish_date missing: {df['publish_date'].isna().mean():.0%}")

In [ ]:
# Last 24 months by quarter, candidate types
pd.crosstab(
    cand["publish_date"].dt.to_period("Q"), cand["property_type"].isin(["Apartment", "House"])
).tail(10)

In [ ]:
# Is a missing date random? Missing share by inferred source and district
missing = cand["publish_date"].isna()
display(missing.groupby(source.loc[cand.index]).mean().round(2))
missing.groupby(cand["district"]).mean().sort_values().round(2).to_frame("date_missing_share").T

## A6 · Location

- How many distinct `District`, `City` and `Town` values? How many missing?
- Any spelling variants (accents, casing, parentheses like `Lagoa (Algarve)`)?
- Is any `City` name used in more than one `District`? What does that mean for grouping by `City` alone?
- How many listings per parish? How many parishes have very few? (This matters for T3's grouped splits.)

In [ ]:
# Location cardinality and gaps
pd.DataFrame(
    {
        "n_unique": cand[["district", "municipality", "parish"]].nunique(),
        "missing": cand[["district", "municipality", "parish"]].isna().sum(),
    }
)

In [ ]:
cand["district"].value_counts().to_frame("n").T

In [ ]:
# Parish names are not unique across municipalities; count (municipality, parish) pairs
pairs = cand.groupby(["municipality", "parish"]).size()
shared = cand.groupby("parish")["municipality"].nunique().gt(1).sum()
print(
    f"parish names used in >1 municipality: {shared}; (municipality, parish) pairs: {len(pairs):,}"
)
print("listings per (municipality, parish):")
pairs.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(1)

In [ ]:
# How many pairs have very few listings? (matters for grouped splits in T3)
pd.cut(
    pairs, [0, 1, 4, 9, 29, 99, np.inf], labels=["1", "2-4", "5-9", "10-29", "30-99", "100+"]
).value_counts().sort_index()

## Findings log

The decisions taken from Part A. Full evidence and row counts: `DATA_CARD.md`.
Code: `src/fairprice/data.py` (`clean`).

| # | Observation (with numbers) | Action |
|---|---|---|
| 1 | Two sources with different fields: 68,247 raw rows lack garage/EV/energy level, and never have bedrooms or dates | `source` column (A / B / AB); bias note |
| 2 | Estate and Farm are mostly land (€72 and €390/m² on total_area); Other - Residential is mixed (€500/m²) | Scope: Apartment, House, Duplex, Studio, Mansion, Manor |
| 3 | 43 rows outside Portugal (Brazil, Dubai) | Drop. Keep Madeira and the Azores |
| 4 | total_area holds lot sizes, placeholders (10,368) and negative values; living_area is 98% filled | Use living_area, 20–2,000 m² |
| 5 | Flats at €600–1,000, €1 house: rents and placeholders | Drop price < €10,000 |
| 6 | €/m² below 100 = rents or typos; above 25k = typos (€2bn house) | Keep €100–25,000/m² |
| 7 | 621 in-scope listings dated before 2024 | Drop; keep undated |
| 8 | 3,020 bathrooms, negative counts, bedrooms capped at 21 | Set to missing, keep the row |
| 9 | 11.5k exact duplicates; +4.9k relisted; ~20k groups of the same home on both portals | Exact, then relisted (keep latest), then merge across sources |
| 10 | In source B, total_rooms ≈ bedrooms + 1 (equal in only 6%) | Defer to T4 (features) |
| 11 | Viseu: 20 listings after cleaning | Bias note |